<a href="https://colab.research.google.com/github/ABINAYA600/GENAI_LAB/blob/main/program_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q torchvision

In [1]:
import torch
import torchvision
import transformers

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("All libraries loaded successfully!")

PyTorch: 2.13.0+cu130
Torchvision: 0.28.0+cu130
Transformers: 4.49.0
All libraries loaded successfully!


In [2]:
# ============================================================
# EXPERIMENT 6
# RETRIEVAL-AUGMENTED GENERATION (RAG) USING FAISS
# ============================================================

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ============================================================
# 1. KNOWLEDGE BASE
# ============================================================

documents = [
    """
    Generative Artificial Intelligence is a branch of AI that creates
    new content such as text, images, audio, video and computer programs.
    """,

    """
    Large Language Models are transformer-based models trained on massive
    text datasets. They are used for text generation, summarization,
    translation, question answering and conversational AI.
    """,

    """
    Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.
    """,

    """
    Vector databases store high-dimensional embeddings and perform
    similarity searches. Examples of vector databases include FAISS,
    ChromaDB, Pinecone, Weaviate and Milvus.
    """,

    """
    Prompt engineering is the process of designing clear instructions
    that guide a language model to produce accurate and useful responses.
    Common techniques include zero-shot, few-shot and role-based prompting.
    """,

    """
    Fine-tuning adapts a pretrained language model to a specific domain
    or task by training it further using a smaller domain-specific dataset.
    """
]


# ============================================================
# 2. LOAD PRETRAINED EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ============================================================
# 3. CREATE DOCUMENT EMBEDDINGS
# ============================================================

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

document_embeddings = document_embeddings.astype(
    "float32"
)


# ============================================================
# 4. NORMALIZE DOCUMENT EMBEDDINGS
# ============================================================

faiss.normalize_L2(
    document_embeddings
)


# ============================================================
# 5. CREATE FAISS VECTOR DATABASE
# ============================================================

embedding_dimension = document_embeddings.shape[1]

vector_database = faiss.IndexFlatIP(
    embedding_dimension
)

vector_database.add(
    document_embeddings
)


# ============================================================
# 6. LOAD PRETRAINED GENERATION MODEL
# ============================================================

print("Loading language model...")

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)


# ============================================================
# 7. RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents(query, top_k=2):

    # Convert query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize query embedding
    faiss.normalize_L2(
        query_embedding
    )

    # Search for similar documents
    similarity_scores, document_indices = vector_database.search(
        query_embedding,
        top_k
    )

    retrieved_documents = []

    for index, score in zip(
        document_indices[0],
        similarity_scores[0]
    ):

        retrieved_documents.append({
            "document": documents[index].strip(),
            "score": float(score)
        })

    return retrieved_documents


# ============================================================
# 8. ANSWER GENERATION FUNCTION
# ============================================================

def generate_answer(query, retrieved_documents):

    # Combine retrieved documents
    context = "\n\n".join(
        item["document"]
        for item in retrieved_documents
    )

    # Create prompt
    prompt = f"""
Answer the question using only the information provided
in the context.

Context:
{context}

Question:
{query}

Instructions:
1. Give a clear and concise answer.
2. Do not add information that is not present in the context.
3. If the answer is unavailable, state:
The answer is not available in the knowledge base.

Answer:
"""

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate answer
    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    # Decode
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


# ============================================================
# 9. RUN RAG SYSTEM
# ============================================================

print("\n")
print("=" * 55)
print("RETRIEVAL-AUGMENTED GENERATION SYSTEM")
print("=" * 55)

user_query = input(
    "\nEnter your question: "
)


# Retrieve relevant documents
retrieved_results = retrieve_documents(
    query=user_query,
    top_k=2
)


# Generate answer
answer = generate_answer(
    query=user_query,
    retrieved_documents=retrieved_results
)


# ============================================================
# 10. DISPLAY RETRIEVED DOCUMENTS
# ============================================================

print("\nRETRIEVED DOCUMENTS")
print("-" * 55)

for number, item in enumerate(
    retrieved_results,
    start=1
):

    print(f"\nDocument {number}:")
    print(item["document"])

    print(
        f"Similarity Score: {item['score']:.4f}"
    )


# ============================================================
# 11. DISPLAY GENERATED ANSWER
# ============================================================

print("\nGENERATED ANSWER")
print("-" * 55)

print(answer)


# ============================================================
# RESULT
# ============================================================

print("\n")
print("=" * 55)
print("RESULT")
print("=" * 55)

print(
    "Thus, a Retrieval-Augmented Generation system was "
    "successfully developed using a pretrained embedding "
    "model, the FAISS vector database, and a transformer-based "
    "language model."
)

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]



RETRIEVAL-AUGMENTED GENERATION SYSTEM

Enter your question: What is Retrieval-Augmented Generation?

RETRIEVED DOCUMENTS
-------------------------------------------------------

Document 1:
Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.
Similarity Score: 0.6933

Document 2:
Generative Artificial Intelligence is a branch of AI that creates
    new content such as text, images, audio, video and computer programs.
Similarity Score: 0.3435

GENERATED ANSWER
-------------------------------------------------------
combines information retrieval with text generation


RESULT
Thus, a Retrieval-Augmented Generation system was successfully developed using a pretrained embedding model, the FAISS vector database, and a transformer-based language model.
